# WLASL EDA Start

Notebook do szybkiej eksploracji metadanych i spojnosc plikow video.

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Notebook is at: szum/notebooks/WLASL_EDA/ -> go 2 levels up to reach szum/
PROJECT_ROOT = Path.cwd().parents[1]

DATASET_DIR = PROJECT_ROOT / 'kaggle_dataset'
META_FILE = DATASET_DIR / 'WLASL_v0.3.json'
VIDEOS_DIR = DATASET_DIR / 'videos'

print('Project root:', PROJECT_ROOT)
print('Metadata exists:', META_FILE.exists())
print('Videos dir exists:', VIDEOS_DIR.exists())

In [ ]:
with META_FILE.open('r', encoding='utf-8') as f:
    records = json.load(f)

print('Entries in WLASL_v0.3:', len(records))
print('Keys in first entry:', list(records[0].keys()))
records[0]

In [ ]:
# Flatten selected metadata for quick statistics
rows = []
for item in records:
    gloss = item.get('gloss')
    for inst in item.get('instances', []):
        rows.append({
            'gloss': gloss,
            'video_id': inst.get('video_id'),
            'split': inst.get('split'),
            'signer_id': inst.get('signer_id'),
        })

df = pd.DataFrame(rows)
df.head()

In [ ]:
print('Unique glosses:', df['gloss'].nunique())
print('Total instances:', len(df))
print('Split distribution:')
print(df['split'].value_counts(dropna=False))

In [ ]:
# Check how many videos are present on disk
if VIDEOS_DIR.exists():
    video_files = {p.stem for p in VIDEOS_DIR.glob('*.mp4')}
    expected = set(df['video_id'].dropna().astype(str).unique())
    missing_local = sorted(expected - video_files)
    print('Expected video IDs:', len(expected))
    print('Found local .mp4 files:', len(video_files))
    print('Missing local videos:', len(missing_local))
    missing_local[:20]
else:
    print('videos directory not found')